# Mirror-map visualizer

Loads a Phase 1 checkpoint and shows, for each patch:
- `x` — original cloud-free S2 (RGB composite)
- `g_phi(x)` — mirror-space image
- `f_psi(g_phi(x))` — round-trip reconstruction
- `g_phi(x) - x` — mirror-map residual (how much it moves each pixel)
- Per-band spectral profiles comparing x, mirror space, and reconstruction

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import torch
import rasterio

# Add phase1 source to path
PHASE1 = Path('phase1_mirror_map')
sys.path.insert(0, str(PHASE1))

from icnn import ICNN, ICNNGradient
from inverse_map import InverseMap

In [ ]:
# ── Config — edit these ────────────────────────────────────────────────────────
CKPT_PATH  = Path('/scratch/oywang/cs159/checkpoints/phase1/spectral_sw0.1_mw1/best.pt')
DATA_ROOT  = Path('data/ROIs1158_spring_s2')   # cloud-free patches
N_PATCHES  = 4                                  # number of patches to visualize
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

# Model hyperparams — must match the training run
N_CHANNELS    = 13
NGF           = 64
N_RES_BLOCKS  = 6
ICNN_FILTERS  = 64
ICNN_LAYERS   = 6
STRONG_CONVEXITY = 0.3

print(f'Device: {DEVICE}')
print(f'Checkpoint: {CKPT_PATH}')

In [ ]:
# ── Build and load models ──────────────────────────────────────────────────────
icnn = ICNN(
    n_channels=N_CHANNELS,
    n_filters=ICNN_FILTERS,
    n_layers=ICNN_LAYERS,
    strong_convexity=STRONG_CONVEXITY,
)
g_phi = ICNNGradient(icnn).to(DEVICE)
f_psi = InverseMap(
    n_channels=N_CHANNELS,
    ngf=NGF,
    n_res_blocks=N_RES_BLOCKS,
).to(DEVICE)

ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
g_phi.load_state_dict(ckpt['g_phi'])
f_psi.load_state_dict(ckpt['f_psi'])
g_phi.eval(); f_psi.eval()
print(f'Loaded checkpoint from epoch {ckpt["epoch"]}  '
      f'(best_val_loss={ckpt["best_val_loss"]:.5f})')

In [ ]:
# ── Load patches ───────────────────────────────────────────────────────────────
tifs = sorted(DATA_ROOT.rglob('*.tif'))[:N_PATCHES]
if not tifs:
    raise FileNotFoundError(f'No .tif files found under {DATA_ROOT}. '
                            'Run `python download_local_data.py` first.')
print(f'Using {len(tifs)} patches: {[t.name for t in tifs]}')

patches = []
for tif in tifs:
    with rasterio.open(tif) as src:
        arr = src.read().astype(np.float32) / 10000.0   # -> [0, 1] reflectance
    patches.append(arr)

x_batch = torch.tensor(np.stack(patches)).to(DEVICE)   # (N, 13, H, W)

In [ ]:
# ── Forward pass ──────────────────────────────────────────────────────────────
with torch.no_grad():
    y_batch     = g_phi(x_batch)           # mirror space
    xr_batch    = f_psi(y_batch)           # reconstruction

x_np  = x_batch.cpu().numpy()             # (N, 13, H, W)
y_np  = y_batch.cpu().numpy()
xr_np = xr_batch.cpu().numpy()

In [ ]:
# ── Helper: 13-band -> display RGB (bands B4/B3/B2 with percentile stretch) ──
def to_rgb(arr13, p=(2, 98)):
    """(13, H, W) float -> (H, W, 3) display RGB."""
    rgb = arr13[[3, 2, 1]].copy()   # R=B4, G=B3, B=B2
    for c in range(3):
        lo, hi = np.percentile(rgb[c], p)
        rgb[c] = np.clip((rgb[c] - lo) / (hi - lo + 1e-8), 0.0, 1.0)
    return np.transpose(rgb, (1, 2, 0))

def diff_display(arr13_a, arr13_b):
    """Signed mean-across-bands difference, clipped for display."""
    d = (arr13_a - arr13_b).mean(axis=0)   # (H, W)
    scale = max(abs(d.min()), abs(d.max()), 1e-8)
    return d / scale, scale

In [ ]:
# ── Panel grid: x | y=g_phi(x) | x_recon | residual y-x ──────────────────────
N = len(tifs)
fig, axes = plt.subplots(N, 4, figsize=(16, 4 * N))
if N == 1:
    axes = axes[np.newaxis, :]

col_titles = [
    r'$x$ (original)',
    r'$g_\phi(x)$ (mirror space)',
    r'$f_\psi(g_\phi(x))$ (reconstruction)',
    r'$g_\phi(x) - x$ (residual)',
]

for row, (xi, yi, xri, name) in enumerate(zip(x_np, y_np, xr_np, tifs)):
    d, scale = diff_display(yi, xi)

    imgs = [to_rgb(xi), to_rgb(yi), to_rgb(xri)]
    for col, img in enumerate(imgs):
        ax = axes[row, col]
        ax.imshow(img)
        ax.axis('off')
        if row == 0:
            ax.set_title(col_titles[col], fontsize=11)
        if col == 0:
            ax.set_ylabel(name.stem, fontsize=9)

    # Residual with diverging colormap
    ax = axes[row, 3]
    im = ax.imshow(d, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.axis('off')
    if row == 0:
        ax.set_title(col_titles[3], fontsize=11)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04,
                 label=f'scale={scale:.4f}')

fig.suptitle('Phase 1 mirror map — spatial visualisation', y=1.01, fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# ── Spectral profiles: per-band spatial mean for x, y, x_recon ───────────────
# Sentinel-2 band names (13 bands, 0-indexed)
BAND_NAMES = ['B1','B2','B3','B4','B5','B6','B7','B8','B8A','B9','B10','B11','B12']

fig, axes = plt.subplots(1, N, figsize=(5 * N, 4), sharey=False)
if N == 1:
    axes = [axes]

x_ticks = range(N_CHANNELS)

for ax, xi, yi, xri, name in zip(axes, x_np, y_np, xr_np, tifs):
    # Spatial mean per band
    mx  = xi.mean(axis=(1, 2))
    my  = yi.mean(axis=(1, 2))
    mxr = xri.mean(axis=(1, 2))

    ax.plot(x_ticks, mx,  'b-o',  ms=4, label=r'$x$')
    ax.plot(x_ticks, my,  'r--s', ms=4, label=r'$g_\phi(x)$')
    ax.plot(x_ticks, mxr, 'g:^',  ms=4, label=r'$f_\psi(g_\phi(x))$')
    ax.set_xticks(list(x_ticks))
    ax.set_xticklabels(BAND_NAMES, rotation=45, fontsize=8)
    ax.set_xlabel('Band')
    ax.set_ylabel('Mean reflectance')
    ax.set_title(name.stem, fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

fig.suptitle('Per-band spectral profiles: original vs mirror vs reconstruction',
             fontsize=12)
fig.tight_layout()
plt.show()

In [ ]:
# ── Residual histogram: how large are the mirror-map displacements? ────────────
residuals = (y_np - x_np).ravel()   # all patches, all bands, all pixels

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(residuals, bins=200, density=True, color='steelblue', alpha=0.8)
ax.axvline(0, color='k', linewidth=0.8, linestyle='--')
ax.set_xlabel(r'$g_\phi(x) - x$ (reflectance units)')
ax.set_ylabel('Density')
ax.set_title('Distribution of mirror-map residuals across all patches and bands')
ax.text(0.98, 0.95,
        f'mean={residuals.mean():.4f}\nstd={residuals.std():.4f}\n'
        f'|max|={np.abs(residuals).max():.4f}',
        transform=ax.transAxes, va='top', ha='right', fontsize=9,
        bbox=dict(boxstyle='round', fc='white', alpha=0.7))
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

# Reconstruction error stats
recon_err = (xr_np - x_np)
mae = np.abs(recon_err).mean()
print(f'Round-trip MAE (f_psi ∘ g_phi):  {mae:.5f}')
print(f'Mirror residual std (g_phi - x): {residuals.std():.5f}')